In [ ]:
import numpy as np
from sim_stim import make_srs

from gould_2026.datasets import Zong22Dataset

from gould_2026.prediction.kalman_filter import StreamingKalmanFilter
from gould_2026.plotting import Palette, LINEWIDTH, paper_plot_context, make_violinplot_inner_kws
from gould_2026.stim_designer import OptimizationMethod
import functools
import pandas as pd
from gould_2026.utils import angle_between
from itertools import cycle
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
stim_magnitude = 10
output_1_latents = None
output_2_traces = None
output_3_stim_pattern = None
output_4_theta_violinplot = None
output_5_show_v = None
output_6_show_s_hat = None

show_v = False

In [ ]:
stim_magnitude = float(stim_magnitude)

zero_thresh = 0.05
amount_to_add = 0
switch_time = 304


In [ ]:
def make_srs_zong(data, rng, n_runs=1, show_tqdm=False, overrides=None, stim_magnitude=10):
    if overrides is None:
        overrides = {}

    common = dict(
        stim_magnitude=stim_magnitude,
        design_method='optimized identity u_to_s',
        exit_time=np.inf,
        stim_rate=None,
        smoothing_tau=1,
        centerer_init_size=8 * 25,
        initial_nostim_period=30,
        regular_stim_iter=cycle([1 / 10, 1 / 3]),
        stim_timing_method='regular',
        autoreg=functools.partial(StreamingKalmanFilter, steps_between_refits=5),
    )

    to_run = {
        'learning from stim': common | dict(attempt_correction=True, heed_stimuli=True),
        'learning from stim random design': common | dict(attempt_correction=True, heed_stimuli=True, design_method='many neurons', optimization_method=OptimizationMethod.CHEAT_HIGHD_VEC_MANY_NEURONS),
        # 'ignoring stim': common | dict(attempt_correction=False, heed_stimuli=True),
        'unaware of stim': common | dict(attempt_correction=False, heed_stimuli=False),
    }

    return make_srs(data, rng, to_run, n_runs=n_runs, show_tqdm=show_tqdm, overrides=overrides)


In [ ]:

d = Zong22Dataset()

rng = np.random.default_rng(0)
data = d.neural_data

srs = make_srs_zong(
    data, rng, n_runs=1, show_tqdm=True,
    overrides=dict(
        stim_magnitude=stim_magnitude,
        regressor_stim_delay=0 * data.dt,
        delay_switch_time=switch_time,
        delay_switch_amount=amount_to_add,
    ),
)

In [ ]:
i = 40
sr = srs['learning from stim'][0]

In [ ]:
with paper_plot_context():
    fig_1, axs1 = plt.subplots(ncols=1, figsize=(2.351,1.854), sharex=False, sharey=False, layout='constrained')

    latents = sr.log['latents'].slice_by_time(slice(30,None))
    axs1.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
    stim_s = sr.log['stim_intended_samples'].t - latents.dt

    l = 1
    r = 4.7
    ax_n = 0
    center_t = sr.log['stim_intended_samples'].t[i]
    latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
    line = axs1.plot(latents[:, 0], latents[:, 1], color='k', lw=2*LINEWIDTH)
    stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
    latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))

    for arrow_index in [17, 50]:
        axs1.annotate('',
                        xytext=(latents[arrow_index, 0], latents[arrow_index, 1]),
                        xy=(latents[arrow_index+1, 0], latents[arrow_index+1, 1]),
                        arrowprops=dict(arrowstyle="simple", color='k'),
                        size=15*LINEWIDTH
                        )

    for j in [0, 1]:
        axs1.plot(latents_s[j, 0], latents_s[j, 1], '.', color='r')

        if show_v:
            axs1.annotate('',
                            xytext=(latents_s[j, 0], latents_s[j, 1]),
                            xy=(latents_s[j, 0] + .5, latents_s[j, 1] + 0),
                            arrowprops=dict(arrowstyle="simple", color=Palette.v),
                            size=15*LINEWIDTH
                            )

    u = sr.stim_designer.log[i]['u']
    idx = np.argsort(np.abs(u))[::-1]
    # n_nonzero = np.linalg.norm(u,ord=0)
    n_nonzero = (np.abs(u) > zero_thresh).sum() # these were actually zeroed out with a custom line, this isn't a threshold
    axs1.axis('off')
    print(f'{n_nonzero=}')

if output_1_latents is not None:
    fig_1.savefig(output_1_latents)

In [ ]:
with paper_plot_context():
    fig_2, axs2 = plt.subplots(ncols=1, figsize=(2.351,1.854), sharex=False, sharey=False, layout='constrained')
    high_d = sr.log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
    axs2.plot(high_d.t, high_d[:,idx[:int(n_nonzero)]], color='k', lw=1/1.5*LINEWIDTH)
    axs2.set_xticks([302, 304, 306,308])
    for stim_t in stim_s:
        axs2.axvline(stim_t, color='r')
    axs2.set_ylim((-1, 11.5))
    axs2.spines[['right', 'top']].set_visible(False)
    axs2.set_xlabel('Time (s)')

if output_2_traces is not None:
    fig_2.savefig(output_2_traces)


In [ ]:
with paper_plot_context():
    fig_3, ax = plt.subplots(ncols=1, figsize=(2.351, 2.351), sharex=False, sharey=False, layout='constrained')

    u[u < zero_thresh] = np.nan
    ax.matshow(-d.ops['meanImg'], cmap='Grays')
    xs, ys = list(zip(*[cell['med'] for cell in d.stat]))
    ax.scatter(np.array(ys)[u > zero_thresh], np.array(xs)[u > zero_thresh], s=15, color='red')
    ax.axis('off')

if output_3_stim_pattern is not None:
    fig_3.savefig(output_3_stim_pattern)


In [ ]:
with paper_plot_context():
    fig_4, ax = plt.subplots(ncols=1, figsize=(1.7, 1.7), sharex=False, sharey=False, layout='constrained')

    log = srs['learning from stim'][0].stim_designer.log
    control_log = srs['learning from stim random design'][0].stim_designer.log

    df = pd.DataFrame({'l':  log + control_log,
                       'condition': ['optimized'] * len(log) + ['random'] * len(control_log)})
    df['v'] = df['l'].apply(lambda x: x['v'])
    df['s_hat'] = df['l'].apply(lambda x: x['observed_s_hat'])

    df['theta'] = df[['v', 's_hat']].apply(lambda x: angle_between(x['v'], x['s_hat']), axis=1)

    sns.violinplot(data=df, x='condition', y='theta', hue='condition', ax=ax, cut=0, palette=[Palette.s_obs, Palette.blind], inner_kws=make_violinplot_inner_kws())
    # sns.stripplot(data=df, x='condition', y='theta', hue='condition', ax=ax, palette=[Palette.s_designed, Palette.blind], s=2)
    ax.set_ylabel('$\\theta$')

    ax.set_xlabel('')
    ax.spines[['right', 'top']].set_visible(False)
    ax.set_yticks([0, 45, 90, 135])


if output_4_theta_violinplot is not None:
    fig_4.savefig(output_4_theta_violinplot)


In [ ]:
with paper_plot_context():
    fig, axs = plt.subplots(ncols=1, figsize=(2.351,1.854), sharex=False, sharey=False, layout='constrained')

    latents = sr.log['latents'].slice_by_time(slice(30,None))
    axs.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
    stim_s = sr.log['stim_intended_samples'].t - 0

    l = 1
    r = -1/10
    ax_n = 0
    center_t = sr.log['stim_intended_samples'].t[i]
    latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
    line = axs.plot(latents[:, 0], latents[:, 1], color='k', lw=2*LINEWIDTH)

    for j in [0]:
        axs.plot(latents[-1, 0], latents[-1, 1], '.', color='r')

        axs.annotate('',
                        xytext=(latents[-1, 0]+.1, latents[-1, 1]-.1),
                        xy=(latents[-1, 0] + .8, latents[-1, 1]-.1),
                        arrowprops=dict(arrowstyle="-|>", color=Palette.v, linewidth=2*LINEWIDTH),
                        size=15*LINEWIDTH
                        )

    axs.axis('off')

if output_5_show_v is not None:
    fig.savefig(output_5_show_v)

In [ ]:
%matplotlib inline
with paper_plot_context():
    fig, axs = plt.subplots(ncols=1, figsize=(2.351,1.854), sharex=False, sharey=False, layout='constrained')

    latents = sr.log['latents'].slice_by_time(slice(30,None))
    axs.plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
    stim_s = sr.log['stim_intended_samples'].t - 0

    l = 1
    r = 3/10
    ax_n = 0
    center_t = sr.log['stim_intended_samples'].t[i]
    latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
    line = axs.plot(latents[:, 0], latents[:, 1], color='k', lw=2*LINEWIDTH)

    last_point = latents[-1, 0:2]

    stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
    latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))


    j = latents.time_to_sample(latents_s.t[0])
    stim_point = latents[j, :2]

    pred_point = latents[j, :2] + np.diff(latents[j-4:j, :2]).mean(axis=0) * 10/10

    axs.plot(*list(zip(stim_point, pred_point)), linestyle=(0,(1,1)), linewidth=2*LINEWIDTH, color=Palette.f_hat)
    axs.plot(*list(zip(pred_point, last_point)), '-', linewidth=2*LINEWIDTH, color=Palette.s_obs)
    axs.plot(*stim_point, '.', color='r')
    axs.plot(*last_point, '.', color='k')


    axs.annotate('',
                    xytext=(pred_point[0]-.1, pred_point[1]),
                    xy=(pred_point[0] + 1.8, pred_point[1]),
                    arrowprops=dict(arrowstyle="-|>", color=Palette.v, linewidth=2*LINEWIDTH),
                    size=15*LINEWIDTH
                    )


    d = last_point - pred_point
    thetas = np.linspace(0,np.atan2(d[1], d[0]), 20)
    arc = np.vstack([np.cos(thetas), np.sin(thetas)]).T*1.4 + pred_point

    plt.plot(arc[:,0], arc[:,1], color='k')

    axs.axis('off')

if output_6_show_s_hat is not None:
    fig.savefig(output_6_show_s_hat)